In [1]:
# =============================================================================
# 1. DATA PREPROCESSING AND FEATURE ENGINEERING FOR ML MODELS
# =============================================================================

print("="*80)
print("MACHINE LEARNING MODELS IMPLEMENTATION")
print("="*80)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

# Advanced ML Libraries
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Time series libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

print("Libraries imported successfully!")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"CatBoost version: {cb.__version__}")

# Load the processed data
print("\nLoading processed data...")
refactored_df = pd.read_csv('../data/refactored_df.csv')
weather_data = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

# Convert date columns
refactored_df['Date'] = pd.to_datetime(refactored_df['Date'])
refactored_df['Branch'].replace({'SBD1': 'VJW'}, inplace=True)
weather_data['Date'] = pd.to_datetime(weather_data['Date'])
trends_df['Date'] = pd.to_datetime(trends_df['Month'])

print(f"Data loaded successfully!")
print(f"Sales data shape: {refactored_df.shape}")
print(f"Weather data shape: {weather_data.shape}")
print(f"Trends data shape: {trends_df.shape}")


MACHINE LEARNING MODELS IMPLEMENTATION
Libraries imported successfully!
LightGBM version: 4.6.0
XGBoost version: 3.1.1
CatBoost version: 1.2.8

Loading processed data...
Data loaded successfully!
Sales data shape: (147595, 9)
Weather data shape: (456, 11)
Trends data shape: (82, 3)


In [ ]:
main_df = (
    refactored_df.groupby([refactored_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
main_df.columns = ['Date', 'Branch', 'Qty']
main_df = main_df.merge(weather_data, on=['Date', 'Branch']).reset_index(drop=True)
main_df = main_df.merge(trends_df, on=['Date'])
main_df = main_df.drop(columns=['Month'])
main_df['Seasonality_Level'] = main_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

In [6]:
new_df = main_df.groupby('Date').agg({'Qty': 'sum', 'Min Temp': 'min', 'Max Temp': 'max', 'Avg Temp': 'mean', 'Min Humidity': 'min', 'Max Humidity': 'max', 'Avg Humidity': 'mean', 'Min Wind Speed': 'min', 'Max Wind Speed': 'max', 'Avg Wind Speed': 'mean', 'Interest': 'mean', 'Seasonality_Level': 'mean'}).reset_index()

In [10]:
# =============================================================================
# 2.1 COMPREHENSIVE FEATURE ENGINEERING
# =============================================================================

from sklearn.preprocessing import LabelEncoder
print("\n2. COMPREHENSIVE FEATURE ENGINEERING")
print("-" * 50)

def drop_nan_columns(df, threshold=1.0, verbose=True):
    """
    Drop columns with a fraction of NaN values above the given threshold.
    """
    nan_ratio = df.isna().mean()
    drop_cols = nan_ratio[nan_ratio >= threshold].index.tolist()
    df_cleaned = df.drop(columns=drop_cols)

    if verbose:
        print(
            f"Dropped {len(drop_cols)} column(s) with ≥{threshold*100:.0f}% NaN values:")
        if drop_cols:
            print(drop_cols)
        else:
            print("No columns dropped.")

    return df_cleaned


def group_expanding_shifted_mean(group):
    """Helper function for expanding mean with shift - reduces overhead when reused"""
    return group.expanding().mean().shift(1)


def group_rolling_shifted_mean(group, window):
    """Helper function for rolling mean with shift - for complete past-only purity"""
    return group.shift(1).rolling(window=window, min_periods=1).mean()


def create_ml_features_leakage_safe(df):
    """
    Create comprehensive features for machine learning models with COMPLETE LEAKAGE-SAFE engineering
    """
    print("Creating comprehensive features with COMPLETE leakage-safe engineering...")

    # Start with the base dataframe
    ml_df = df.copy()
    ml_df = drop_nan_columns(ml_df, 0.5)

    # Check data availability for each branch
    # branch_counts = ml_df.groupby('Branch').size()
    # print(f"  Data points per branch: {dict(branch_counts)}")

    # CRITICAL: Sort by date and branch FIRST for proper temporal ordering
    ml_df = ml_df.sort_values(['Date']).reset_index(drop=True)

    # Cache grouped objects for efficiency
    # branch_qty_group = ml_df.groupby('Branch')['Qty']
    # branch_temp_group = ml_df.groupby('Branch')['Avg Temp']
    # branch_humidity_group = ml_df.groupby('Branch')['Avg Humidity']
    # branch_wind_group = ml_df.groupby('Branch')['Avg Wind Speed']

    # 1. TIME-BASED FEATURES
    print("  Creating time-based features...")
    ml_df['year'] = ml_df['Date'].dt.year
    ml_df['month'] = ml_df['Date'].dt.month
    # ml_df['day'] = ml_df['Date'].dt.day
    # ml_df['dayofweek'] = ml_df['Date'].dt.dayofweek
    # ml_df['dayofyear'] = ml_df['Date'].dt.dayofyear
    # ml_df['week'] = ml_df['Date'].dt.isocalendar().week.astype(int)
    # ml_df['quarter'] = ml_df['Date'].dt.quarter

    # Cyclical encoding for time features
    ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
    ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
    # ml_df['dayofweek_sin'] = np.sin(2 * np.pi * ml_df['dayofweek'] / 7)
    # ml_df['dayofweek_cos'] = np.cos(2 * np.pi * ml_df['dayofweek'] / 7)
    # ml_df['quarter_sin'] = np.sin(2 * np.pi * ml_df['quarter'] / 4)
    # ml_df['quarter_cos'] = np.cos(2 * np.pi * ml_df['quarter'] / 4)

    # 2. LAG FEATURES (LEAKAGE-SAFE)
    # print("  Creating lag features...")
    # lag_periods = [1, 2, 3, 6, 12]  # months instead of days
    # for lag in lag_periods:
    #     ml_df[f'qty_lag_{lag}m'] = branch_qty_group.shift(lag)
    #     # FIXED: Use groupby().apply() for proper branch boundaries
    #     if lag <= 3:  # Only for short lags
    #         lag_val = lag  # Capture loop variable
    #         ml_df[f'qty_lag_{lag}m_mean'] = (
    #             branch_qty_group
    #             .apply(lambda x: x.shift(lag_val).rolling(window=min(3, lag_val), min_periods=1).mean())
    #             .reset_index(level=0, drop=True)
    #         )

    # 3. ROLLING STATISTICS (LEAKAGE-SAFE)
    # print("  Creating rolling statistics...")
    # rolling_windows = [2, 3, 6, 12]  # months
    # for window in rolling_windows:
    #     # FIXED: Use groupby().apply() for proper branch boundaries
    #     # OPTIONAL: For complete past-only purity, uncomment the shift(1) version
    #     window_val = window  # Capture loop variable
    #     ml_df[f'qty_rolling_mean_{window}m'] = (
    #         branch_qty_group
    #         .apply(lambda x: x.rolling(window=window_val, min_periods=1).mean())
    #         .reset_index(level=0, drop=True)
    #     )
    #     # For complete past-only: .apply(lambda x: x.shift(1).rolling(window=window_val, min_periods=1).mean())

    #     ml_df[f'qty_rolling_std_{window}m'] = (
    #         branch_qty_group
    #         .apply(lambda x: x.rolling(window=window_val, min_periods=1).std())
    #         .reset_index(level=0, drop=True)
    #     )
    #     ml_df[f'qty_rolling_max_{window}m'] = (
    #         branch_qty_group
    #         .apply(lambda x: x.rolling(window=window_val, min_periods=1).max())
    #         .reset_index(level=0, drop=True)
    #     )
    #     ml_df[f'qty_rolling_min_{window}m'] = (
    #         branch_qty_group
    #         .apply(lambda x: x.rolling(window=window_val, min_periods=1).min())
    #         .reset_index(level=0, drop=True)
    #     )
    #     ml_df[f'qty_rolling_sum_{window}m'] = (
    #         branch_qty_group
    #         .apply(lambda x: x.rolling(window=window_val, min_periods=1).sum())
    #         .reset_index(level=0, drop=True)
    #     )

    # 4. EXPANDING STATISTICS (LEAKAGE-SAFE - SHIFT INSIDE GROUP)
    print("  Creating expanding statistics...")
    # FIXED: Use helper function for efficiency
    # ml_df['qty_expanding_mean'] = (
    #     branch_qty_group
    #     .apply(group_expanding_shifted_mean)
    #     .reset_index(level=0, drop=True)
    # )
    # ml_df['qty_expanding_std'] = (
    #     branch_qty_group
    #     .apply(lambda x: x.expanding().std().shift(1))
    #     .reset_index(level=0, drop=True)
    # )
    # ml_df['qty_expanding_max'] = (
    #     branch_qty_group
    #     .apply(lambda x: x.expanding().max().shift(1))
    #     .reset_index(level=0, drop=True)
    # )
    # ml_df['qty_expanding_min'] = (
    #     branch_qty_group
    #     .apply(lambda x: x.expanding().min().shift(1))
    #     .reset_index(level=0, drop=True)
    # )

    # OPTIONAL ENHANCEMENTS: Additional useful features
    print("  Creating additional enhancement features...")
    # Rate of change (lag diff)
    # ml_df['qty_diff_1m'] = branch_qty_group.diff(1)
    # ml_df['qty_diff_3m'] = branch_qty_group.diff(3)

    # # Previous year same month lag
    # ml_df['qty_lag_12m'] = branch_qty_group.shift(12)

    # # Sales volatility (rolling std shifted)
    # ml_df['qty_volatility_3m'] = (
    #     branch_qty_group
    #     .apply(lambda x: x.rolling(3).std().shift(1))
    #     .reset_index(level=0, drop=True)
    # )

    # 5. SEASONAL FEATURES (LEAKAGE-SAFE - PER-BRANCH-PER-MONTH)
    print("  Creating leakage-safe seasonal features...")
    # FIXED: Compute per-branch-per-month expanding means
    ml_df['monthly_seasonality'] = (
        ml_df.groupby(['month'])['Qty']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )

    # ml_df['quarterly_seasonality'] = (
    #     ml_df.groupby(['Branch', 'quarter'])['Qty']
    #     .apply(lambda x: x.expanding().mean().shift(1))
    #     .reset_index(level=[0, 1], drop=True)
    # )

    # ml_df['dow_seasonality'] = (
    #     ml_df.groupby(['Branch', 'dayofweek'])['Qty']
    #     .apply(lambda x: x.expanding().mean().shift(1))
    #     .reset_index(level=[0, 1], drop=True)
    # )

    # 6. WEATHER LAG FEATURES (LEAKAGE-SAFE)
    # print("  Creating weather lag features...")
    # weather_lags = [1, 2, 3, 6]  # months
    # for lag in weather_lags:
    #     ml_df[f'temp_lag_{lag}m'] = branch_temp_group.shift(lag)
    #     ml_df[f'humidity_lag_{lag}m'] = branch_humidity_group.shift(lag)
    #     ml_df[f'wind_lag_{lag}m'] = branch_wind_group.shift(lag)

    # 7. TRENDS LAG FEATURES (LEAKAGE-SAFE - GROUP BY BRANCH IF NEEDED)
    # print("  Creating trends lag features...")
    # trends_lags = [1, 2, 3, 6]
    # for lag in trends_lags:
    #     # Check if Interest varies by Branch
    #     if ml_df['Interest'].nunique().max() > 1:
    #         # Interest varies by branch, so group by Branch
    #         ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)
    #     else:
    #         # Interest is global, so shift globally
    #         ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)

    # 8. PRODUCT FEATURES (LABEL ENCODER MOVED TO AFTER SPLIT)
    print("  Creating product features...")
    # NOTE: LabelEncoder will be fitted after train/test split to avoid leakage
    # For now, just create a placeholder
    ml_df['branch_encoded'] = 0  # Will be properly encoded after split

    # 9. INTERACTION FEATURES
    print("  Creating interaction features...")
    ml_df['temp_humidity_interaction'] = ml_df['Avg Temp'] * \
        ml_df['Avg Humidity']
    ml_df['temp_wind_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Wind Speed']

    # 10. STATISTICAL FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe statistical features...")
    # Temperature statistics
    ml_df['temp_range'] = ml_df['Max Temp'] - ml_df['Min Temp']
    ml_df['humidity_range'] = ml_df['Max Humidity'] - ml_df['Min Humidity']
    ml_df['wind_range'] = ml_df['Max Wind Speed'] - ml_df['Min Wind Speed']

    # FIXED: Temperature deviation using expanding mean + shift per branch-month
    ml_df['temp_monthly_mean_past'] = (
        ml_df.groupby(['month'])['Avg Temp']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['temp_deviation'] = ml_df['Avg Temp'] - \
        ml_df['temp_monthly_mean_past']

    # FIXED: Humidity deviation using expanding mean + shift per branch-month
    ml_df['humidity_monthly_mean_past'] = (
        ml_df.groupby(['month'])['Avg Humidity']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['humidity_deviation'] = ml_df['Avg Humidity'] - \
        ml_df['humidity_monthly_mean_past']

    # 11. BUSINESS FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe business features...")
    # Days since last sale (convert to months for monthly data)
    ml_df['months_since_last_sale'] = ml_df['Date'].diff().dt.days / 30.44

    # # FIXED: Sales momentum using shifted expanding mean
    # ml_df['sales_momentum_3m'] = np.where(
    #     ml_df['qty_expanding_mean'] != 0,
    #     ml_df['qty_rolling_mean_3m'] / ml_df['qty_expanding_mean'],
    #     1.0
    # )
    # ml_df['sales_momentum_6m'] = np.where(
    #     ml_df['qty_expanding_mean'] != 0,
    #     ml_df['qty_rolling_mean_6m'] / ml_df['qty_expanding_mean'],
    #     1.0
    # )

    # FIXED: Market share using proper denominator alignment
    # Compute total quantity per Date first
    date_totals = ml_df.groupby(
        'Date')['Qty'].sum().reset_index(name='total_qty')

    # Merge and shift total_qty globally by chronological order
    date_totals = date_totals.sort_values('Date')
    date_totals['total_qty_prev'] = date_totals['total_qty'].shift(1)

    # Merge back into ml_df
    # ml_df = ml_df.merge(
    #     date_totals[['Date', 'total_qty_prev']], on='Date', how='left')
    # ml_df['prev_branch_qty'] = branch_qty_group.shift(1)
    # ml_df['prev_market_share'] = np.where(
    #     ml_df['total_qty_prev'] > 0,
    #     ml_df['prev_branch_qty'] / ml_df['total_qty_prev'],
    #     0
    # )
    # Clean up temporary columns
    # ml_df.drop(['total_qty_prev', 'prev_branch_qty'], axis=1, inplace=True)

    # 12. HANDLE REMAINING NaN VALUES (OPTIMIZED)
    print("  Handling remaining NaN values...")

    # FIXED: Vectorized NaN filling for efficiency with safety checks
    fill_dict = {}

    # Safe median calculation
    qty_median = ml_df['Qty'].median() if 'Qty' in ml_df.columns else 0

    # Fill lag features with 0
    lag_cols = [col for col in ml_df.columns if 'lag' in col]
    for col in lag_cols:
        fill_dict[col] = 0

    # Fill rolling features
    rolling_cols = [col for col in ml_df.columns if 'rolling' in col]
    for col in rolling_cols:
        if col.endswith('_mean') or col.endswith('_sum'):
            fill_dict[col] = qty_median
        else:
            fill_dict[col] = ml_df[col].median() if col in ml_df.columns else 0

    # Fill expanding features
    expanding_cols = [col for col in ml_df.columns if 'expanding' in col]
    for col in expanding_cols:
        fill_dict[col] = qty_median

    # Fill seasonal features
    seasonal_cols = [col for col in ml_df.columns if 'seasonality' in col]
    for col in seasonal_cols:
        fill_dict[col] = qty_median

    # Fill deviation features
    deviation_cols = [col for col in ml_df.columns if 'deviation' in col]
    for col in deviation_cols:
        fill_dict[col] = 0.0

    # Fill diff features
    diff_cols = [col for col in ml_df.columns if 'diff' in col]
    for col in diff_cols:
        fill_dict[col] = 0.0

    # Fill volatility features
    volatility_cols = [col for col in ml_df.columns if 'volatility' in col]
    for col in volatility_cols:
        fill_dict[col] = 0.0

    # Apply all fills at once
    ml_df.fillna(fill_dict, inplace=True)

    # Fill any remaining NaN values
    remaining_nan_cols = ml_df.columns[ml_df.isnull().any()].tolist()
    if remaining_nan_cols:
        print(f"    Filling remaining {len(remaining_nan_cols)} columns...")
        for col in remaining_nan_cols:
            if ml_df[col].dtype in ['int64', 'float64']:
                ml_df[col] = ml_df[col].fillna(ml_df[col].median())
            else:
                ml_df[col] = ml_df[col].fillna(
                    ml_df[col].mode()[0] if not ml_df[col].mode().empty else 0)

    print("  COMPLETE leakage-safe feature engineering completed!")
    print(f"  Total features created: {ml_df.shape[1]}")

    return ml_df

# Apply feature engineering
ml_df = create_ml_features_leakage_safe(new_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df.columns if any(x in col for x in ['market_share', 'days_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df.isnull().sum()
missing_percentage = (missing_values / len(ml_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0].head(10))



2. COMPREHENSIVE FEATURE ENGINEERING
--------------------------------------------------
Creating comprehensive features with COMPLETE leakage-safe engineering...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Creating time-based features...
  Creating expanding statistics...
  Creating additional enhancement features...
  Creating leakage-safe seasonal features...
  Creating product features...
  Creating interaction features...
  Creating leakage-safe statistical features...
  Creating leakage-safe business features...
  Handling remaining NaN values...
    Filling remaining 3 columns...
  COMPLETE leakage-safe feature engineering completed!
  Total features created: 29

Feature Categories:
  Time-based: 8 features
  Lag features: 0 features
  Rolling statistics: 0 features
  Expanding statistics: 0 features
  Weather features: 9 features
  Trends features: 1 features
  Product features: 1 features
  Interaction features: 2 features
  Statistical features: 5 features

In [3]:
# =============================================================================
# 2. COMPREHENSIVE FEATURE ENGINEERING
# =============================================================================

from sklearn.preprocessing import LabelEncoder
print("\n2. COMPREHENSIVE FEATURE ENGINEERING")
print("-" * 50)

def drop_nan_columns(df, threshold=1.0, verbose=True):
    """
    Drop columns with a fraction of NaN values above the given threshold.
    """
    nan_ratio = df.isna().mean()
    drop_cols = nan_ratio[nan_ratio >= threshold].index.tolist()
    df_cleaned = df.drop(columns=drop_cols)

    if verbose:
        print(
            f"Dropped {len(drop_cols)} column(s) with ≥{threshold*100:.0f}% NaN values:")
        if drop_cols:
            print(drop_cols)
        else:
            print("No columns dropped.")

    return df_cleaned


def group_expanding_shifted_mean(group):
    """Helper function for expanding mean with shift - reduces overhead when reused"""
    return group.expanding().mean().shift(1)


def group_rolling_shifted_mean(group, window):
    """Helper function for rolling mean with shift - for complete past-only purity"""
    return group.shift(1).rolling(window=window, min_periods=1).mean()


def create_ml_features_leakage_safe(df):
    """
    Create comprehensive features for machine learning models with COMPLETE LEAKAGE-SAFE engineering
    """
    print("Creating comprehensive features with COMPLETE leakage-safe engineering...")

    # Start with the base dataframe
    ml_df = df.copy()
    ml_df = drop_nan_columns(ml_df, 0.5)

    # Check data availability for each branch
    branch_counts = ml_df.groupby('Branch').size()
    print(f"  Data points per branch: {dict(branch_counts)}")

    # CRITICAL: Sort by date and branch FIRST for proper temporal ordering
    ml_df = ml_df.sort_values(['Branch', 'Date']).reset_index(drop=True)

    # Cache grouped objects for efficiency
    branch_qty_group = ml_df.groupby('Branch')['Qty']
    branch_temp_group = ml_df.groupby('Branch')['Avg Temp']
    branch_humidity_group = ml_df.groupby('Branch')['Avg Humidity']
    branch_wind_group = ml_df.groupby('Branch')['Avg Wind Speed']

    # 1. TIME-BASED FEATURES
    print("  Creating time-based features...")
    ml_df['year'] = ml_df['Date'].dt.year
    ml_df['month'] = ml_df['Date'].dt.month
    # ml_df['day'] = ml_df['Date'].dt.day
    # ml_df['dayofweek'] = ml_df['Date'].dt.dayofweek
    # ml_df['dayofyear'] = ml_df['Date'].dt.dayofyear
    # ml_df['week'] = ml_df['Date'].dt.isocalendar().week.astype(int)
    # ml_df['quarter'] = ml_df['Date'].dt.quarter

    # Cyclical encoding for time features
    ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
    ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
    # ml_df['dayofweek_sin'] = np.sin(2 * np.pi * ml_df['dayofweek'] / 7)
    # ml_df['dayofweek_cos'] = np.cos(2 * np.pi * ml_df['dayofweek'] / 7)
    # ml_df['quarter_sin'] = np.sin(2 * np.pi * ml_df['quarter'] / 4)
    # ml_df['quarter_cos'] = np.cos(2 * np.pi * ml_df['quarter'] / 4)

    # 2. LAG FEATURES (LEAKAGE-SAFE)
    print("  Creating lag features...")
    lag_periods = [1, 2, 3, 6, 12]  # months instead of days
    for lag in lag_periods:
        ml_df[f'qty_lag_{lag}m'] = branch_qty_group.shift(lag)
        # FIXED: Use groupby().apply() for proper branch boundaries
        if lag <= 3:  # Only for short lags
            lag_val = lag  # Capture loop variable
            ml_df[f'qty_lag_{lag}m_mean'] = (
                branch_qty_group
                .apply(lambda x: x.shift(lag_val).rolling(window=min(3, lag_val), min_periods=1).mean())
                .reset_index(level=0, drop=True)
            )

    # 3. ROLLING STATISTICS (LEAKAGE-SAFE)
    print("  Creating rolling statistics...")
    rolling_windows = [2, 3, 6, 12]  # months
    for window in rolling_windows:
        # FIXED: Use groupby().apply() for proper branch boundaries
        # OPTIONAL: For complete past-only purity, uncomment the shift(1) version
        window_val = window  # Capture loop variable
        ml_df[f'qty_rolling_mean_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )
        # For complete past-only: .apply(lambda x: x.shift(1).rolling(window=window_val, min_periods=1).mean())

        ml_df[f'qty_rolling_std_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).std())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_max_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).max())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_min_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).min())
            .reset_index(level=0, drop=True)
        )
        ml_df[f'qty_rolling_sum_{window}m'] = (
            branch_qty_group
            .apply(lambda x: x.rolling(window=window_val, min_periods=1).sum())
            .reset_index(level=0, drop=True)
        )

    # 4. EXPANDING STATISTICS (LEAKAGE-SAFE - SHIFT INSIDE GROUP)
    print("  Creating expanding statistics...")
    # FIXED: Use helper function for efficiency
    ml_df['qty_expanding_mean'] = (
        branch_qty_group
        .apply(group_expanding_shifted_mean)
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_std'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().std().shift(1))
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_max'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().max().shift(1))
        .reset_index(level=0, drop=True)
    )
    ml_df['qty_expanding_min'] = (
        branch_qty_group
        .apply(lambda x: x.expanding().min().shift(1))
        .reset_index(level=0, drop=True)
    )

    # OPTIONAL ENHANCEMENTS: Additional useful features
    print("  Creating additional enhancement features...")
    # Rate of change (lag diff)
    ml_df['qty_diff_1m'] = branch_qty_group.diff(1)
    ml_df['qty_diff_3m'] = branch_qty_group.diff(3)

    # Previous year same month lag
    ml_df['qty_lag_12m'] = branch_qty_group.shift(12)

    # Sales volatility (rolling std shifted)
    ml_df['qty_volatility_3m'] = (
        branch_qty_group
        .apply(lambda x: x.rolling(3).std().shift(1))
        .reset_index(level=0, drop=True)
    )

    # 5. SEASONAL FEATURES (LEAKAGE-SAFE - PER-BRANCH-PER-MONTH)
    print("  Creating leakage-safe seasonal features...")
    # FIXED: Compute per-branch-per-month expanding means
    ml_df['monthly_seasonality'] = (
        ml_df.groupby(['Branch', 'month'])['Qty']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )

    # ml_df['quarterly_seasonality'] = (
    #     ml_df.groupby(['Branch', 'quarter'])['Qty']
    #     .apply(lambda x: x.expanding().mean().shift(1))
    #     .reset_index(level=[0, 1], drop=True)
    # )

    # ml_df['dow_seasonality'] = (
    #     ml_df.groupby(['Branch', 'dayofweek'])['Qty']
    #     .apply(lambda x: x.expanding().mean().shift(1))
    #     .reset_index(level=[0, 1], drop=True)
    # )

    # 6. WEATHER LAG FEATURES (LEAKAGE-SAFE)
    print("  Creating weather lag features...")
    weather_lags = [1, 2, 3, 6]  # months
    for lag in weather_lags:
        ml_df[f'temp_lag_{lag}m'] = branch_temp_group.shift(lag)
        ml_df[f'humidity_lag_{lag}m'] = branch_humidity_group.shift(lag)
        ml_df[f'wind_lag_{lag}m'] = branch_wind_group.shift(lag)

    # 7. TRENDS LAG FEATURES (LEAKAGE-SAFE - GROUP BY BRANCH IF NEEDED)
    print("  Creating trends lag features...")
    trends_lags = [1, 2, 3, 6]
    for lag in trends_lags:
        # Check if Interest varies by Branch
        if ml_df.groupby('Branch')['Interest'].nunique().max() > 1:
            # Interest varies by branch, so group by Branch
            ml_df[f'trends_lag_{lag}m'] = ml_df.groupby(
                'Branch')['Interest'].shift(lag)
        else:
            # Interest is global, so shift globally
            ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)

    # 8. PRODUCT FEATURES (LABEL ENCODER MOVED TO AFTER SPLIT)
    print("  Creating product features...")
    # NOTE: LabelEncoder will be fitted after train/test split to avoid leakage
    # For now, just create a placeholder
    ml_df['branch_encoded'] = 0  # Will be properly encoded after split

    # 9. INTERACTION FEATURES
    print("  Creating interaction features...")
    ml_df['temp_humidity_interaction'] = ml_df['Avg Temp'] * \
        ml_df['Avg Humidity']
    ml_df['temp_wind_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Wind Speed']

    # 10. STATISTICAL FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe statistical features...")
    # Temperature statistics
    ml_df['temp_range'] = ml_df['Max Temp'] - ml_df['Min Temp']
    ml_df['humidity_range'] = ml_df['Max Humidity'] - ml_df['Min Humidity']
    ml_df['wind_range'] = ml_df['Max Wind Speed'] - ml_df['Min Wind Speed']

    # FIXED: Temperature deviation using expanding mean + shift per branch-month
    ml_df['temp_monthly_mean_past'] = (
        ml_df.groupby(['Branch', 'month'])['Avg Temp']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['temp_deviation'] = ml_df['Avg Temp'] - \
        ml_df['temp_monthly_mean_past']

    # FIXED: Humidity deviation using expanding mean + shift per branch-month
    ml_df['humidity_monthly_mean_past'] = (
        ml_df.groupby(['Branch', 'month'])['Avg Humidity']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    ml_df['humidity_deviation'] = ml_df['Avg Humidity'] - \
        ml_df['humidity_monthly_mean_past']

    # 11. BUSINESS FEATURES (LEAKAGE-SAFE)
    print("  Creating leakage-safe business features...")
    # Days since last sale (convert to months for monthly data)
    ml_df['months_since_last_sale'] = ml_df.groupby(
        'Branch')['Date'].diff().dt.days / 30.44

    # # FIXED: Sales momentum using shifted expanding mean
    # ml_df['sales_momentum_3m'] = np.where(
    #     ml_df['qty_expanding_mean'] != 0,
    #     ml_df['qty_rolling_mean_3m'] / ml_df['qty_expanding_mean'],
    #     1.0
    # )
    # ml_df['sales_momentum_6m'] = np.where(
    #     ml_df['qty_expanding_mean'] != 0,
    #     ml_df['qty_rolling_mean_6m'] / ml_df['qty_expanding_mean'],
    #     1.0
    # )

    # FIXED: Market share using proper denominator alignment
    # Compute total quantity per Date first
    date_totals = ml_df.groupby(
        'Date')['Qty'].sum().reset_index(name='total_qty')

    # Merge and shift total_qty globally by chronological order
    date_totals = date_totals.sort_values('Date')
    date_totals['total_qty_prev'] = date_totals['total_qty'].shift(1)

    # Merge back into ml_df
    ml_df = ml_df.merge(
        date_totals[['Date', 'total_qty_prev']], on='Date', how='left')
    ml_df['prev_branch_qty'] = branch_qty_group.shift(1)
    ml_df['prev_market_share'] = np.where(
        ml_df['total_qty_prev'] > 0,
        ml_df['prev_branch_qty'] / ml_df['total_qty_prev'],
        0
    )
    # Clean up temporary columns
    ml_df.drop(['total_qty_prev', 'prev_branch_qty'], axis=1, inplace=True)

    # 12. HANDLE REMAINING NaN VALUES (OPTIMIZED)
    print("  Handling remaining NaN values...")

    # FIXED: Vectorized NaN filling for efficiency with safety checks
    fill_dict = {}

    # Safe median calculation
    qty_median = ml_df['Qty'].median() if 'Qty' in ml_df.columns else 0

    # Fill lag features with 0
    lag_cols = [col for col in ml_df.columns if 'lag' in col]
    for col in lag_cols:
        fill_dict[col] = 0

    # Fill rolling features
    rolling_cols = [col for col in ml_df.columns if 'rolling' in col]
    for col in rolling_cols:
        if col.endswith('_mean') or col.endswith('_sum'):
            fill_dict[col] = qty_median
        else:
            fill_dict[col] = ml_df[col].median() if col in ml_df.columns else 0

    # Fill expanding features
    expanding_cols = [col for col in ml_df.columns if 'expanding' in col]
    for col in expanding_cols:
        fill_dict[col] = qty_median

    # Fill seasonal features
    seasonal_cols = [col for col in ml_df.columns if 'seasonality' in col]
    for col in seasonal_cols:
        fill_dict[col] = qty_median

    # Fill deviation features
    deviation_cols = [col for col in ml_df.columns if 'deviation' in col]
    for col in deviation_cols:
        fill_dict[col] = 0.0

    # Fill diff features
    diff_cols = [col for col in ml_df.columns if 'diff' in col]
    for col in diff_cols:
        fill_dict[col] = 0.0

    # Fill volatility features
    volatility_cols = [col for col in ml_df.columns if 'volatility' in col]
    for col in volatility_cols:
        fill_dict[col] = 0.0

    # Apply all fills at once
    ml_df.fillna(fill_dict, inplace=True)

    # Fill any remaining NaN values
    remaining_nan_cols = ml_df.columns[ml_df.isnull().any()].tolist()
    if remaining_nan_cols:
        print(f"    Filling remaining {len(remaining_nan_cols)} columns...")
        for col in remaining_nan_cols:
            if ml_df[col].dtype in ['int64', 'float64']:
                ml_df[col] = ml_df[col].fillna(ml_df[col].median())
            else:
                ml_df[col] = ml_df[col].fillna(
                    ml_df[col].mode()[0] if not ml_df[col].mode().empty else 0)

    print("  COMPLETE leakage-safe feature engineering completed!")
    print(f"  Total features created: {ml_df.shape[1]}")

    return ml_df

# Apply feature engineering
ml_df = create_ml_features_leakage_safe(main_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df.columns if any(x in col for x in ['market_share', 'days_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df.isnull().sum()
missing_percentage = (missing_values / len(ml_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0].head(10))



2. COMPREHENSIVE FEATURE ENGINEERING
--------------------------------------------------
Creating comprehensive features with COMPLETE leakage-safe engineering...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Data points per branch: {'BLR': np.int64(59), 'COK': np.int64(50), 'MAA': np.int64(59), 'SBD': np.int64(59), 'VJW': np.int64(59)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating additional enhancement features...
  Creating leakage-safe seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating leakage-safe statistical features...
  Creating leakage-safe business features...
  Handling remaining NaN values...
    Filling remaining 4 columns...
  COMPLETE leakage-safe feature engineering completed!
  Total features created: 82

Feature Categories:
  Time-based: 8 feat

In [12]:
def fit_label_encoder_on_train(train_data, test_data, val_data=None):
    """
    Fit LabelEncoder only on training data to avoid leakage
    """
    print("Fitting LabelEncoder on training data only...")

    # Fit encoder on training data
    branch_encoder = LabelEncoder()
    train_data.loc[:, 'branch_encoded'] = branch_encoder.fit_transform(
        train_data['Branch'])

    # Transform test and validation data using fitted encoder
    test_data.loc[:, 'branch_encoded'] = branch_encoder.transform(
        test_data['Branch'])
    if val_data is not None:
        val_data.loc[:, 'branch_encoded'] = branch_encoder.transform(
            val_data['Branch'])

    return train_data, test_data, val_data, branch_encoder

In [13]:
# =============================================================================
# 3. DATA PREPARATION AND TRAIN-TEST SPLIT
# =============================================================================

print("\n3. DATA PREPARATION AND TRAIN-TEST SPLIT")

def prepare_ml_data(df, target_col='Qty', test_size=0.2, validation_size=0.1):
    """
    Prepare data for machine learning models with improved NaN handling
    """
    print("Preparing data for ML models...")
    
    
    # Remove rows with missing target values
    df_clean = df.dropna(subset=[target_col]).copy()
    print(f"  After removing missing target values: {len(df_clean)} samples")
    
    # Check for any remaining missing values
    missing_before = df_clean.isnull().sum().sum()
    print(f"  Missing values before cleaning: {missing_before}")
    
    # Fill any remaining missing values with appropriate strategies
    print("  Filling remaining missing values...")
    
    # Fill numerical columns with median
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if col != target_col and df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            print(f"    Filled {col} with median: {df_clean[col].median():.2f}")
    
    # Fill categorical columns with mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"    Filled {col} with mode: {mode_val}")
    
    # Define feature columns (exclude target and non-predictive columns)
    exclude_cols = [target_col, 'Date', 'Year', 'Month', 'Week', 'Branch', 'Segment', 'Rating']
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Total samples: {len(df_clean)}")
    
    # Check final missing values
    missing_after = df_clean.isnull().sum().sum()
    print(f"  Missing values after cleaning: {missing_after}")
    
    if missing_after > 0:
        print("  Warning: Still have missing values!")
        missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
        print(f"  Columns with missing values: {missing_cols}")
    
    # Sort by date
    df_clean_sorted = df_clean.sort_values('Date')
    
    # Calculate split indices
    total_samples = len(df_clean_sorted)
    # Create splits
    train_data = df_clean_sorted[df_clean_sorted['Date'] < '2023-04-01']
    test_data = df_clean_sorted[df_clean_sorted['Date'] >= '2023-04-01']
    val_data = df_clean_sorted[(df_clean_sorted['Date'] >= '2023-11-01') & (df_clean_sorted['Date'] < '2024-04-01')]
    # train_data, val_data, test_data, branch_encoder = fit_label_encoder_on_train(train_data, val_data, test_data)
    
    
    # Extract features and targets for each split
    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_val = val_data[feature_cols]
    y_val = val_data[target_col]
    X_test = test_data[feature_cols]
    y_test = test_data[target_col]
    
    print(f"  Train set: {len(X_train)} samples ({len(X_train)/total_samples*100:.1f}%)")
    print(f"  Validation set: {len(X_val)} samples ({len(X_val)/total_samples*100:.1f}%)")
    print(f"  Test set: {len(X_test)} samples ({len(X_test)/total_samples*100:.1f}%)")
    
    # Date ranges for each split
    print(f"  Train period: {train_data['Date'].min()} to {train_data['Date'].max()}")
    print(f"  Validation period: {val_data['Date'].min()} to {val_data['Date'].max()}")
    print(f"  Test period: {test_data['Date'].min()} to {test_data['Date'].max()}")
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'feature_cols': feature_cols,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data,
        # 'branch_encoder': branch_encoder
    }

# Prepare the data
ml_data = prepare_ml_data(ml_df)

# Display data summary
print(f"\nData Summary:")
print(f"  Features: {len(ml_data['feature_cols'])}")
print(f"  Training samples: {len(ml_data['X_train'])}")
print(f"  Validation samples: {len(ml_data['X_val'])}")
print(f"  Test samples: {len(ml_data['X_test'])}")

# Check for any remaining missing values
print(f"\nMissing Values Check:")
print(f"  Train set missing values: {ml_data['X_train'].isnull().sum().sum()}")
print(f"  Validation set missing values: {ml_data['X_val'].isnull().sum().sum()}")
print(f"  Test set missing values: {ml_data['X_test'].isnull().sum().sum()}")

# Display feature importance preview (using correlation with target)
print(f"\nTop 10 Features by Correlation with Target:")
correlations = ml_data['X_train'].corrwith(ml_data['y_train']).abs().sort_values(ascending=False)
print(correlations.head(10))

# Save the prepared data for later use
print(f"\nSaving prepared data...")
ml_data['X_train'].to_csv('../data/ml_X_train.csv', index=False)
ml_data['X_val'].to_csv('../data/ml_X_val.csv', index=False)
ml_data['X_test'].to_csv('../data/ml_X_test.csv', index=False)
ml_data['y_train'].to_csv('../data/ml_y_train.csv', index=False)
ml_data['y_val'].to_csv('../data/ml_y_val.csv', index=False)
ml_data['y_test'].to_csv('../data/ml_y_test.csv', index=False)

print("Data preparation completed successfully!")



3. DATA PREPARATION AND TRAIN-TEST SPLIT
Preparing data for ML models...
  After removing missing target values: 59 samples
  Missing values before cleaning: 0
  Filling remaining missing values...
  Total features: 27
  Total samples: 59
  Missing values after cleaning: 0
  Train set: 47 samples (79.7%)
  Validation set: 5 samples (8.5%)
  Test set: 12 samples (20.3%)
  Train period: 2019-04-01 00:00:00 to 2023-03-01 00:00:00
  Validation period: 2023-11-01 00:00:00 to 2024-03-01 00:00:00
  Test period: 2023-04-01 00:00:00 to 2024-03-01 00:00:00

Data Summary:
  Features: 27
  Training samples: 47
  Validation samples: 5
  Test samples: 12

Missing Values Check:
  Train set missing values: 0
  Validation set missing values: 0
  Test set missing values: 0

Top 10 Features by Correlation with Target:
Seasonality_Level            0.805572
month_sin                    0.641337
temp_humidity_interaction    0.634916
Min Humidity                 0.627138
humidity_range               0.62713

In [31]:
ml_data['X_train'].columns

Index(['Min Temp', 'Max Temp', 'Avg Temp', 'Min Humidity', 'Max Humidity',
       'Avg Humidity', 'Min Wind Speed', 'Max Wind Speed', 'Avg Wind Speed',
       'Interest', 'Seasonality_Level', 'year', 'month', 'month_sin',
       'month_cos', 'qty_lag_1m', 'qty_lag_1m_mean', 'qty_lag_2m',
       'qty_lag_2m_mean', 'qty_lag_3m', 'qty_lag_3m_mean', 'qty_lag_6m',
       'qty_lag_12m', 'qty_rolling_mean_2m', 'qty_rolling_std_2m',
       'qty_rolling_max_2m', 'qty_rolling_min_2m', 'qty_rolling_sum_2m',
       'qty_rolling_mean_3m', 'qty_rolling_std_3m', 'qty_rolling_max_3m',
       'qty_rolling_min_3m', 'qty_rolling_sum_3m', 'qty_rolling_mean_6m',
       'qty_rolling_std_6m', 'qty_rolling_max_6m', 'qty_rolling_min_6m',
       'qty_rolling_sum_6m', 'qty_rolling_mean_12m', 'qty_rolling_std_12m',
       'qty_rolling_max_12m', 'qty_rolling_min_12m', 'qty_rolling_sum_12m',
       'qty_expanding_mean', 'qty_expanding_std', 'qty_expanding_max',
       'qty_expanding_min', 'qty_diff_1m', 'qty_diff_3

In [14]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)


In [15]:
# =============================================================================
# 4. LIGHTGBM MODEL IMPLEMENTATION
# =============================================================================

print("\n4. LIGHTGBM MODEL IMPLEMENTATION")
print("-" * 50)

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Calculate comprehensive evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

def train_lightgbm_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train LightGBM model with hyperparameter tuning"""
    
    print("Training LightGBM model...")
    
    # 1. Baseline LightGBM Model
    print("  Training baseline LightGBM model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'random_state': 42
    }
    
    # Create datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Train baseline model
    baseline_model = lgb.train(
        baseline_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "LightGBM Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "LightGBM Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "LightGBM Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'num_leaves': [31, 50, 100, 200],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'feature_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_fraction': [0.8, 0.9, 0.95, 1.0],
        'bagging_freq': [5, 10, 15],
        'min_child_samples': [20, 30, 50],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0]
    }
    
    # Use RandomizedSearchCV for efficiency
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        metric='mae',
        boosting_type='gbdt',
        verbose=-1,
        random_state=42,
        n_estimators=1000
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        lgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=tscv,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = lgb.train(
        final_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "LightGBM Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "LightGBM Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "LightGBM Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importance(importance_type='gain')
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train LightGBM model
lgb_results = train_lightgbm_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nLightGBM model training completed successfully!")
print(f"Total training time: {sum(lgb_results['training_times'].values()):.2f} seconds")



4. LIGHTGBM MODEL IMPLEMENTATION
--------------------------------------------------
Training LightGBM model...
  Training baseline LightGBM model...
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[195]	valid_0's l1: 19954.2
    Baseline training time: 0.29 seconds
    Baseline validation RMSE: 26295.4255
    Baseline validation R²: -0.5798
  Performing hyperparameter tuning...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Training until validation scores don't improve for 100 rounds
Early stopp

In [16]:
# =============================================================================
# 5. XGBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n5. XGBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_xgboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train XGBoost model with hyperparameter tuning"""
    
    print("Training XGBoost model...")
    
    # 1. Baseline XGBoost Model
    print("  Training baseline XGBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 1000,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'verbosity': 0
    }
    
    # Train baseline model
    baseline_model = xgb.XGBRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "XGBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "XGBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "XGBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'max_depth': [3, 4, 5, 6, 7, 8],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'n_estimators': [500, 800, 1000, 1200],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bynode': [0.6, 0.7, 0.8, 0.9, 1.0],
        'reg_alpha': [0, 0.1, 0.5, 1.0],
        'reg_lambda': [0, 0.1, 0.5, 1.0, 2.0],
        'gamma': [0, 0.1, 0.5, 1.0],
        'min_child_weight': [1, 3, 5, 7]
    }
    
    # Use RandomizedSearchCV for efficiency
    xgb_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        eval_metric='mae',
        random_state=42,
        verbosity=0
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        xgb_model,
        param_grid,
        n_iter=50,  # Number of parameter settings sampled
        cv=tscv,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=[(X_val, y_val)],
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = xgb.XGBRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "XGBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "XGBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "XGBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    # 6. Advanced XGBoost Features
    print("  Training advanced XGBoost model with additional features...")
    start_time = time.time()
    
    # Advanced parameters for better performance
    advanced_params = final_params.copy()
    advanced_params.update({
        'tree_method': 'hist',  # Use histogram-based algorithm
        'grow_policy': 'lossguide',  # Grow policy for better performance
        'max_leaves': 0,  # Let max_depth control tree size
        'max_bin': 256,  # Number of bins for histogram
        'predictor': 'cpu_predictor',  # Use CPU predictor
        'enable_categorical': False,  # Disable categorical features
        'interaction_constraints': None,  # No interaction constraints
        'monotone_constraints': None,  # No monotone constraints
    })
    
    # Train advanced model
    advanced_model = xgb.XGBRegressor(**advanced_params)
    advanced_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    advanced_time = time.time() - start_time
    
    # Make predictions with advanced model
    advanced_train_pred = advanced_model.predict(X_train)
    advanced_val_pred = advanced_model.predict(X_val)
    advanced_test_pred = advanced_model.predict(X_test)
    
    # Calculate advanced metrics
    advanced_train_metrics = calculate_metrics(y_train, advanced_train_pred, "XGBoost Advanced Train")
    advanced_val_metrics = calculate_metrics(y_val, advanced_val_pred, "XGBoost Advanced Val")
    advanced_test_metrics = calculate_metrics(y_test, advanced_test_pred, "XGBoost Advanced Test")
    
    print(f"    Advanced training time: {advanced_time:.2f} seconds")
    print(f"    Advanced validation RMSE: {advanced_val_metrics['RMSE']:.4f}")
    print(f"    Advanced validation R²: {advanced_val_metrics['R2']:.4f}")
    
    # Compare all XGBoost models
    print("  All XGBoost models comparison:")
    print(f"    Baseline RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Final RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Advanced RMSE: {advanced_val_metrics['RMSE']:.4f}")
    
    best_model = 'Advanced' if advanced_val_metrics['RMSE'] < final_val_metrics['RMSE'] else 'Final'
    print(f"    Best XGBoost model: {best_model}")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'advanced_model': advanced_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'advanced_metrics': {
            'train': advanced_train_metrics,
            'val': advanced_val_metrics,
            'test': advanced_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            },
            'advanced': {
                'train': advanced_train_pred,
                'val': advanced_val_pred,
                'test': advanced_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time,
            'advanced': advanced_time
        }
    }

# Train XGBoost model
xgb_results = train_xgboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nXGBoost model training completed successfully!")
print(f"Total training time: {sum(xgb_results['training_times'].values()):.2f} seconds")



5. XGBOOST MODEL IMPLEMENTATION
--------------------------------------------------
Training XGBoost model...
  Training baseline XGBoost model...
    Baseline training time: 1.45 seconds
    Baseline validation RMSE: 24595.2994
    Baseline validation R²: -0.3821
  Performing hyperparameter tuning...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
    Best parameters: {'subsample': 0.8, 'reg_lambda': 0, 'reg_alpha': 1.0, 'n_estimators': 1000, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.7, 'colsample_bynode': 0.7, 'colsample_bylevel': 0.7}
    Tuning time: 3.84 seconds
  Training final model with best parameters...
    Final training time: 0.57 seconds
    Final validation RMSE: 24190.1396
    Final validation R²: -0.3370
  Analyzing feature importance...
    Top 10 most important features:
      1. Seasonality_Level: 0.4313
      2. humidity_range: 0.1125
      3. temp_range: 0.0674
      4. month_sin: 0.0657
      5. 

In [17]:
# =============================================================================
# 6. CATBOOST MODEL IMPLEMENTATION
# =============================================================================

print("\n6. CATBOOST MODEL IMPLEMENTATION")
print("-" * 50)

def train_catboost_model(X_train, y_train, X_val, y_val, X_test, y_test, feature_cols):
    """Train CatBoost model with hyperparameter tuning"""
    
    print("Training CatBoost model...")
    
    # 1. Baseline CatBoost Model
    print("  Training baseline CatBoost model...")
    start_time = time.time()
    
    baseline_params = {
        'iterations': 1000,
        'learning_rate': 0.1,
        'depth': 6,
        'l2_leaf_reg': 3,
        'bootstrap_type': 'Bayesian',
        'random_seed': 42,
        'od_type': 'Iter',
        'od_wait': 100,
        'verbose': False
    }
    
    # Train baseline model
    baseline_model = cb.CatBoostRegressor(**baseline_params)
    baseline_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    baseline_time = time.time() - start_time
    
    # Make predictions
    baseline_train_pred = baseline_model.predict(X_train)
    baseline_val_pred = baseline_model.predict(X_val)
    baseline_test_pred = baseline_model.predict(X_test)
    
    # Calculate metrics
    baseline_train_metrics = calculate_metrics(y_train, baseline_train_pred, "CatBoost Baseline Train")
    baseline_val_metrics = calculate_metrics(y_val, baseline_val_pred, "CatBoost Baseline Val")
    baseline_test_metrics = calculate_metrics(y_test, baseline_test_pred, "CatBoost Baseline Test")
    
    print(f"    Baseline training time: {baseline_time:.2f} seconds")
    print(f"    Baseline validation RMSE: {baseline_val_metrics['RMSE']:.4f}")
    print(f"    Baseline validation R²: {baseline_val_metrics['R2']:.4f}")
    
    # 2. Hyperparameter Tuning
    print("  Performing hyperparameter tuning...")
    start_time = time.time()
    
    # Define parameter grid for tuning
    param_grid = {
        'iterations': [500, 800, 1000, 1200],
        'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
        'depth': [4, 5, 6, 7, 8],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'bootstrap_type': ['Bayesian', 'Bernoulli'],
        'bagging_temperature': [0, 0.5, 1.0],
        'random_strength': [0, 1, 2],
        'one_hot_max_size': [2, 10, 20],
        'leaf_estimation_method': ['Newton', 'Gradient'],
        'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide']
    }
    
    # Use RandomizedSearchCV for efficiency
    cb_model = cb.CatBoostRegressor(
        random_seed=42,
        od_type='Iter',
        od_wait=100,
        verbose=False
    )
    
    # Randomized search
    random_search = RandomizedSearchCV(
        cb_model,
        param_grid,
        n_iter=30,  # Number of parameter settings sampled
        cv=tscv,  # 3-fold cross-validation
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    
    # Fit the model
    random_search.fit(X_train, y_train, 
                     eval_set=(X_val, y_val),
                     early_stopping_rounds=100,
                     verbose=False)
    
    tuning_time = time.time() - start_time
    
    # Get best parameters
    best_params = random_search.best_params_
    print(f"    Best parameters: {best_params}")
    print(f"    Tuning time: {tuning_time:.2f} seconds")
    
    # 3. Train Final Model with Best Parameters
    print("  Training final model with best parameters...")
    start_time = time.time()
    
    # Update parameters with best found
    final_params = baseline_params.copy()
    final_params.update(best_params)
    
    # Train final model
    final_model = cb.CatBoostRegressor(**final_params)
    final_model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    final_time = time.time() - start_time
    
    # Make predictions with final model
    final_train_pred = final_model.predict(X_train)
    final_val_pred = final_model.predict(X_val)
    final_test_pred = final_model.predict(X_test)
    
    # Calculate final metrics
    final_train_metrics = calculate_metrics(y_train, final_train_pred, "CatBoost Final Train")
    final_val_metrics = calculate_metrics(y_val, final_val_pred, "CatBoost Final Val")
    final_test_metrics = calculate_metrics(y_test, final_test_pred, "CatBoost Final Test")
    
    print(f"    Final training time: {final_time:.2f} seconds")
    print(f"    Final validation RMSE: {final_val_metrics['RMSE']:.4f}")
    print(f"    Final validation R²: {final_val_metrics['R2']:.4f}")
    
    # 4. Feature Importance Analysis
    print("  Analyzing feature importance...")
    feature_importance = final_model.feature_importances_
    feature_names = feature_cols
    
    # Create feature importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"    Top 10 most important features:")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows()):
        print(f"      {i+1}. {row['feature']}: {row['importance']:.4f}")
    
    # 5. Model Comparison
    print("  Model performance comparison:")
    print(f"    Baseline vs Final Model:")
    print(f"      Validation RMSE: {baseline_val_metrics['RMSE']:.4f} → {final_val_metrics['RMSE']:.4f}")
    print(f"      Validation R²: {baseline_val_metrics['R2']:.4f} → {final_val_metrics['R2']:.4f}")
    
    improvement_rmse = ((baseline_val_metrics['RMSE'] - final_val_metrics['RMSE']) / baseline_val_metrics['RMSE']) * 100
    improvement_r2 = ((final_val_metrics['R2'] - baseline_val_metrics['R2']) / abs(baseline_val_metrics['R2'])) * 100
    
    print(f"      RMSE improvement: {improvement_rmse:.2f}%")
    print(f"      R² improvement: {improvement_r2:.2f}%")
    
    return {
        'baseline_model': baseline_model,
        'final_model': final_model,
        'best_params': best_params,
        'baseline_metrics': {
            'train': baseline_train_metrics,
            'val': baseline_val_metrics,
            'test': baseline_test_metrics
        },
        'final_metrics': {
            'train': final_train_metrics,
            'val': final_val_metrics,
            'test': final_test_metrics
        },
        'feature_importance': importance_df,
        'predictions': {
            'baseline': {
                'train': baseline_train_pred,
                'val': baseline_val_pred,
                'test': baseline_test_pred
            },
            'final': {
                'train': final_train_pred,
                'val': final_val_pred,
                'test': final_test_pred
            }
        },
        'training_times': {
            'baseline': baseline_time,
            'tuning': tuning_time,
            'final': final_time
        }
    }

# Train CatBoost model
cb_results = train_catboost_model(
    ml_data['X_train'], ml_data['y_train'],
    ml_data['X_val'], ml_data['y_val'],
    ml_data['X_test'], ml_data['y_test'],
    ml_data['feature_cols']
)

print("\nCatBoost model training completed successfully!")
print(f"Total training time: {sum(cb_results['training_times'].values()):.2f} seconds")



6. CATBOOST MODEL IMPLEMENTATION
--------------------------------------------------
Training CatBoost model...
  Training baseline CatBoost model...
    Baseline training time: 0.27 seconds
    Baseline validation RMSE: 26440.6755
    Baseline validation R²: -0.5973
  Performing hyperparameter tuning...
Fitting 3 folds for each of 30 candidates, totalling 90 fits
    Best parameters: {'random_strength': 1, 'one_hot_max_size': 10, 'learning_rate': 0.15, 'leaf_estimation_method': 'Gradient', 'l2_leaf_reg': 9, 'iterations': 800, 'grow_policy': 'SymmetricTree', 'depth': 4, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.5}
    Tuning time: 5.32 seconds
  Training final model with best parameters...
    Final training time: 0.05 seconds
    Final validation RMSE: 24354.9231
    Final validation R²: -0.3553
  Analyzing feature importance...
    Top 10 most important features:
      1. Seasonality_Level: 34.3348
      2. temp_range: 8.7902
      3. humidity_deviation: 8.2310
      4. 

In [6]:
# =============================================================================
# 7. FORCAST NEXT 12 MONTHS
# =============================================================================


last_date = main_df['Date'].max()
branches = main_df['Branch'].unique()
print("Last available month:", last_date)
print("Branches:", branches)

future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(),
    periods=12, freq='MS'
)


future_df = pd.DataFrame([(d, b) for d in future_dates for b in branches],
                         columns=['Date', 'Branch'])
future_df = future_df.merge(weather_data, on=['Date', 'Branch'], how='left')
future_df = future_df.merge(trends_df, on=['Date'], how='left')

# Fill missing future external data with mean or last available value
for col in ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Interest']:
    if col in future_df.columns:
        future_df[col] = future_df[col].fillna(main_df[col].mean())

# Add seasonality level
future_df['Seasonality_Level'] = future_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

# Append future_df to history
combined_df = pd.concat([main_df, future_df], ignore_index=True)
combined_features = create_ml_features_leakage_safe(combined_df)

# Keep only the new future months
future_features = combined_features[(combined_features['Date'] >= '2023-04-01') & (combined_features['Date'] < '2024-04-01')].copy().reset_index(drop=True)
X_future = future_features[ml_data['feature_cols']].fillna(0)

Last available month: 2024-03-01 00:00:00
Branches: ['BLR' 'MAA' 'SBD' 'VJW' 'COK']
Creating comprehensive features with COMPLETE leakage-safe engineering...
Dropped 1 column(s) with ≥50% NaN values:
['Month']
  Data points per branch: {'BLR': np.int64(71), 'COK': np.int64(62), 'MAA': np.int64(71), 'SBD': np.int64(71), 'VJW': np.int64(71)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating additional enhancement features...
  Creating leakage-safe seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating leakage-safe statistical features...
  Creating leakage-safe business features...
  Handling remaining NaN values...
    Filling remaining 7 columns...
  COMPLETE leakage-safe feature engineering completed!
  Total features created: 95


In [7]:
future_features

,Date,Branch,Qty,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,...,humidity_range,wind_range,temp_monthly_mean_past,temp_deviation,humidity_monthly_mean_past,humidity_deviation,months_since_last_sale,sales_momentum_3m,sales_momentum_6m,prev_market_share
0,2023-04-01,BLR,3119.0,19.0,35.0,26.615278,10.0,100.0,53.258333,0.0,...,90.0,27.7,27.183002,-0.567724,54.505424,-1.247091,1.018397,1.831944,1.634287,0.110520
1,2023-05-01,BLR,2624.0,20.0,34.0,26.012634,25.0,100.0,68.594086,0.0,...,75.0,46.4,26.334813,-0.322178,70.227118,-1.633032,0.985545,1.612557,1.665932,0.073684
2,2023-06-01,BLR,1880.5,20.0,33.0,25.147222,38.0,100.0,72.245833,3.6,...,62.0,41.0,24.726064,0.421158,78.009899,-5.764065,1.018397,1.239945,1.524945,0.079592
3,2023-07-01,BLR,1309.5,20.0,30.0,23.154839,48.0,100.0,79.709677,3.6,...,52.0,41.0,23.779144,-0.624306,83.102059,-3.392382,0.985545,0.947196,1.375421,0.100720
4,2023-08-01,BLR,2268.0,19.0,32.0,24.649194,38.0,100.0,69.420699,1.8,...,62.0,35.3,23.456070,1.193124,84.309571,-14.888872,1.018397,0.895518,1.256359,0.115665
5,2023-09-01,BLR,2841.0,19.0,30.0,23.633056,48.0,100.0,78.537500,3.6,...,52.0,29.9,23.482960,0.150095,84.488346,-5.950846,1.018397,1.050761,1.149395,0.117131
6,2023-10-01,BLR,2246.0,16.0,30.0,24.041129,22.0,100.0,68.670699,0.0,...,78.0,29.5,23.007291,1.033838,85.427279,-16.756580,0.985545,1.195159,1.069956,0.134352
7,2023-11-01,BLR,2646.5,16.0,30.0,22.469444,34.0,100.0,79.108333,0.0,...,66.0,27.7,21.764317,0.705127,85.815026,-6.706693,1.018397,1.254460,1.069904,0.105379
8,2023-12-01,BLR,3284.0,13.0,29.0,21.308737,19.0,100.0,76.924731,3.6,...,81.0,22.3,20.712550,0.596187,82.197831,-5.273100,0.985545,1.319413,1.177572,0.135506
9,2024-01-01,BLR,4325.0,15.0,30.0,21.848118,25.0,100.0,66.743280,3.6,...,75.0,28.1,21.154307,0.693811,72.004982,-5.261702,1.018397,1.637647,1.406064,0.057296


In [38]:
# =============================================================================
# 7. FORECAST NEXT 12 MONTHS - ONE MONTH AT A TIME
# =============================================================================

import pandas as pd
import numpy as np

# Get last available date and branches
last_date = main_df['Date'].max()
branches = main_df['Branch'].unique()
print("Last available month:", last_date)
print("Branches:", branches)

# Generate next 12 month starts
future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(),
    periods=12, freq='MS'
)

# Initialize dataframe for recursive prediction
combined_df = main_df.copy()
predictions_list = []

# Predict month-by-month recursively
for forecast_date in future_dates:
    print(f"\nPredicting for: {forecast_date.strftime('%Y-%m')}")

    # Build future month base
    future_batch = pd.DataFrame({'Date': [forecast_date] * len(branches), 'Branch': branches})

    # Merge external data
    future_batch = future_batch.merge(weather_data, on=['Date', 'Branch'], how='left')
    future_batch = future_batch.merge(trends_df, on=['Date'], how='left')

    # Fill missing values
    for col in ['Avg Temp', 'Avg Humidity', 'Avg Wind Speed', 'Interest']:
        if col in future_batch.columns:
            future_batch[col] = future_batch[col].fillna(main_df[col].mean())

    # Add seasonality
    future_batch['Seasonality_Level'] = future_batch['Date'].dt.month_name().str[:3].map({
        'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
        'May': 2, 'Jan': 2,
        'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
    })

    # Append to combined_df temporarily to compute lag features properly
    temp_df = pd.concat([combined_df, future_batch], ignore_index=True)

    # Generate features including lagged ones
    temp_features = create_ml_features_leakage_safe(temp_df)

    # Extract just the current month's feature rows
    future_features = temp_features[temp_features['Date'] == forecast_date].copy()
    X_future = future_features[ml_data['feature_cols']].fillna(0)

    # Predict using trained models
    cat_pred = cb_results['final_model'].predict(X_future)
    xgb_pred = xgb_results['advanced_model'].predict(X_future)
    lgb_pred = lgb_results['final_model'].predict(X_future)

    # Average ensemble or keep separate
    ensemble_pred = (cat_pred + xgb_pred + lgb_pred) / 3

    # Store predictions
    future_batch['Pred_CatBoost'] = cat_pred
    future_batch['Pred_XGBoost'] = xgb_pred
    future_batch['Pred_LightGBM'] = lgb_pred
    future_batch['Pred_Ensemble'] = ensemble_pred

    predictions_list.append(future_batch[['Date', 'Branch', 'Pred_Ensemble',
                                          'Pred_CatBoost', 'Pred_XGBoost', 'Pred_LightGBM']])

    # Append predicted quantities back to combined_df for next iteration
    append_df = future_batch[['Date', 'Branch']].copy()
    append_df['Qty'] = ensemble_pred  # use ensemble or choose one model
    combined_df = pd.concat([combined_df, append_df], ignore_index=True)

# Combine all month-by-month predictions
future_predictions_df = pd.concat(predictions_list, ignore_index=True)

# Show total forecast
print("\nTotal forecasted quantity (ensemble):", future_predictions_df['Pred_Ensemble'].sum())


Last available month: 2024-03-01 00:00:00
Branches: ['BLR' 'MAA' 'SBD' 'VJW' 'COK']

Predicting for: 2024-04
Creating comprehensive features with COMPLETE leakage-safe engineering...
Dropped 1 column(s) with ≥50% NaN values:
['Month']
  Data points per branch: {'BLR': np.int64(60), 'COK': np.int64(51), 'MAA': np.int64(60), 'SBD': np.int64(60), 'VJW': np.int64(60)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating additional enhancement features...
  Creating leakage-safe seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating leakage-safe statistical features...
  Creating leakage-safe business features...
  Handling remaining NaN values...
    Filling remaining 5 columns...
  COMPLETE leakage-safe feature engineering completed!
  Total features created: 82

Predicting for: 2024-05
Creati